# Stage 3 — campaign summary (pooled across all folders)

Pools **every** `blade_campaign*` folder on Drive (including the duplicate `(1)`/`(2)` folders from the coordination bug), dedups by design hash, and reports the evaluated designs, objective stats, the **optimization progression** (running-best J_fan in evaluation order — rising means it was learning, flat means random), and the Pareto front. Read-only; it does not touch the running campaign.

## 1. Repo + Drive

In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path
IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git","clone","-b","main","https://github.com/clingergab/fan-optimization.git",str(REPO)],check=True)
    else:
        subprocess.run(["git","-C",str(REPO),"pull","origin","main"],check=True)
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = REPO / "data"
for p in (str(REPO), str(REPO/"src")):
    if p not in sys.path: sys.path.insert(0, p)
print("repo:", REPO, "| drive:", DRIVE_ROOT)

## 2. Folders + shards (this also exposes the duplicate-folder split)

In [ ]:
from fanopt.bo.campaign_analysis import find_shards
# Surface EVERY blade_campaign* folder (incl. Drive's duplicate "(1)"/"(2)") and its shards.
byfolder = find_shards(DRIVE_ROOT)
print(f"{len(byfolder)} campaign folder(s) found:")
all_shards = []
for d, shards in byfolder.items():
    print(f"  {d}")
    for s in shards:
        n = sum(1 for l in open(s) if l.strip())
        print(f"      {Path(s).name}: {n} rows")
        all_shards.append(s)
print(f"\n{len(all_shards)} shard files total -> pooling + deduping them all below.")

## 3. Summary — designs, averages, progression, best

In [ ]:
from fanopt.bo.campaign_analysis import campaign_report
r = campaign_report(all_shards)
print(f"unique designs evaluated : {r['unique_designs']}   "
      f"(finite {r['finite']}, failed/NaN {r['failed_nan']}, duplicate rows {r['duplicate_rows']})")
print(f"sources                  : {r['sources']}")
if r.get("j_fan"):
    print(f"J_fan  min/mean/max      : {r['j_fan']['min']:+.2e} / {r['j_fan']['mean']:+.2e} / {r['j_fan']['max']:+.2e}")
    print(f"mass   min/mean/max (g)  : {r['mass_g']['min']:.0f} / {r['mass_g']['mean']:.0f} / {r['mass_g']['max']:.0f}")
    print(f"Sobol(DoE) best J_fan    : {r['sobol_best']:+.2e}")
    print(f"BO best J_fan            : {r['bo_best']:+.2e}   "
          f"(BO {'BEAT' if r['bo_best'] and r['sobol_best'] and r['bo_best']>r['sobol_best'] else 'did NOT beat'} the random DoE)")
    print(f"new running-bests        : {r['progression']['n_new_bests']} over {r['finite']} evals")
    print(f"Pareto front size        : {r['pareto_count']}")
    print("\ntop designs by J_fan:")
    for d in r["top_by_j_fan"]:
        print(f"  J_fan={d['j_fan']:+.2e}  mass={d['mass_g']:.0f}g  {d['source']:5}  {d['design_hash']}")

## 4. Progression plot

In [ ]:
import matplotlib.pyplot as plt
rb = r["progression"]["running_best"]
plt.figure(figsize=(8,4))
plt.plot(range(1, len(rb)+1), rb, marker=".")
plt.xlabel("evaluation # (time order, pooled across all sessions)")
plt.ylabel("running-best J_fan")
plt.title("Optimization progression — rising = learning, flat = random")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 5. Pareto front (the trade-off frontier)

In [ ]:
print(f"Pareto front ({r['pareto_count']} non-dominated designs) — the J_fan vs mass vs deflection trade:")
for d in r["pareto"]:
    print(f"  J_fan={d['j_fan']:+.2e}  mass={d['mass_g']:6.0f} g  defl={d['deflection_mm']:.3f} mm  blades={d['blade_count']}  {d['design_hash']}")